# Predicting Heart Disease competition
> Author | Lavrov Evgeniy - ResInfesser

Models: lightgbm

## 1. Import Libraries & Load Data

In [1]:
import kagglehub
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import roc_auc_score, mean_squared_error
import optuna
import json

/home/koting/_code/Kaggle-Solutions/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.competition_download("playground-series-s6e2")
train_df = pd.read_csv(f"{path}/train.csv")
train_df.head(10)

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence
5,5,38,1,4,138,283,0,0,147,1,1.6,2,2,7,Presence
6,6,59,1,4,130,246,0,2,152,0,0.8,2,2,3,Presence
7,7,60,0,3,120,245,0,0,151,0,1.2,1,0,3,Absence
8,8,48,0,4,140,212,0,2,125,0,0.0,1,0,3,Absence
9,9,44,0,4,150,197,0,0,150,0,0.0,2,0,3,Absence


In [3]:
test_df = pd.read_csv(f"{path}/test.csv")
test_df.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7
3,630003,44,0,3,112,177,0,0,168,0,0.9,1,0,3
4,630004,43,1,1,138,267,0,0,163,0,1.8,2,0,7


## 2. Processing dataframes

In [4]:
train_df = train_df.copy()
train_df = train_df.drop('id', axis=1)

mapping = {
    'Heart Disease': {'Presence':1,'Absence':0},
}

for col, m in mapping.items():
    if col in train_df.columns:
        train_df[col] = train_df[col].map(m)

y_train = train_df['Heart Disease']
train_df = train_df.drop('Heart Disease', axis=1)

X_train = train_df.to_numpy()
X_train_part, X_val, y_train_part, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

In [5]:
test_df = test_df.copy()
ids = test_df['id'].copy() 
test_df = test_df.drop('id', axis=1) 

X_test = test_df.to_numpy()

## 4. Model

### 4.1 Training a model

In [ ]:
def optimize_lgbm_regression(X, y, n_trials=10):
    if hasattr(X, 'values'): X = X.values
    if hasattr(y, 'values'): y = y.values

    # subsampling to reduce search load and duration
    X, _, y, _ = train_test_split(X, y, train_size=0.3, random_state=42)
    
    def objective(trial):
        params = {
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 1.0, log=True),
            'n_estimators': 300,
            'random_state': 42,
            'verbosity': -1,
        }
        model = LGBMRegressor(**params)
        kf = KFold(3, shuffle=True, random_state=42)
        scores = []
        for train_idx, val_idx in kf.split(X):
            X_t, X_v = X[train_idx], X[val_idx]
            y_t, y_v = y[train_idx], y[val_idx]
            model.fit(X_t, y_t, eval_set=[(X_v, y_v)], eval_metric='rmse')
            preds = model.predict(X_v)
            scores.append(-np.sqrt(mean_squared_error(y_v, preds)))
        return np.mean(scores)
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, n_jobs=-1, show_progress_bar=True)


    return study.best_params

# Get predictions
# long search. Because of this, json with parameters is loaded. 
# If it is not there, uncomment the call and get the config.

# best_params = optimize_lgbm_regression(X_train, y_train)

# Save parameters
# with open('predicting-heart-disease-best_params.json', 'w') as f:
#     json.dump(best_params, f, indent=4)

# Loading parameters
with open('predicting-heart-disease-best_params.json', 'r') as f:
    best_params = json.load(f)

model = LGBMRegressor(**best_params, n_estimators=2000, random_state=42)
model.fit(X_train, y_train)
part_preds = model.predict(X_train_part)
val_preds = model.predict(X_val)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019228 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 417
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 13
[LightGBM] [Info] Start training from score 0.448340


/home/koting/_code/Kaggle-Solutions/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/koting/_code/Kaggle-Solutions/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


### 4.2 Use of metrics to assess quality

In [7]:
print(f"Val ROC-AUC: {roc_auc_score(y_val, val_preds):.3f}")
print(f"Part ROC-AUC: {roc_auc_score(y_train_part, part_preds):.3f}")

Val ROC-AUC: 0.961
Part ROC-AUC: 0.961


## 4.3 Retrain for all data

In [8]:
assert X_train.shape[1] == X_test.shape[1] # checking for same number of columns

final_model = LGBMRegressor(**best_params, n_estimators=2000, random_state=42)
final_model.fit(X_train, y_train)

final_preds = final_model.predict(X_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018315 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 417
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 13
[LightGBM] [Info] Start training from score 0.448340


/home/koting/_code/Kaggle-Solutions/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## 5. Final csv

In [9]:
submission = pd.DataFrame({
    'id': ids,
    'Heart Disease': final_preds
})

submission.to_csv('../submissions/predicting-heart-disease-submission.csv', index=False)
print(submission.head())

       id  Heart Disease
0  630000       0.940077
1  630001       0.024188
2  630002       1.006722
3  630003       0.006904
4  630004       0.219571
